# Installations and Imported Packages

In [1]:
# !pip install -q transformers datasets wandb

In [2]:
import os
import time
import random
import glob

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import librosa
import librosa.display

import torch
import torch.nn as nn
import torch.optim as optim
import torchaudio
import torchaudio.transforms as T
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader

from kaggle_secrets import UserSecretsClient
import wandb

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score, confusion_matrix, classification_report, accuracy_score

from pathlib import Path
from tqdm import tqdm

from transformers import AutoFeatureExtractor, ASTForAudioClassification, get_cosine_schedule_with_warmup

import warnings
warnings.filterwarnings("ignore")

print(time.ctime())
print("All libraries are imported")

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

Wed Mar 25 11:17:03 2026
All libraries are imported


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


# User Defined Control Variable

Setting the flag variable **execute_milestone_solutions_code** as **False** ignore the exection of the cells containing the milestone solution code

In [4]:
execute_milestone_solutions_code=True

In [5]:
execute_milestone_solutions_code=False
print(execute_milestone_solutions_code)

False


# Weights & Biases Setup and Test run

In [6]:
secret_label = "WANDB_API_KEY" 
secret_value = UserSecretsClient().get_secret(secret_label)
os.environ['WANDB_API_KEY'] = secret_value
wandb.login()

wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


True

In [7]:
if execute_milestone_solutions_code:
    for run in range(5):
        # Start a new wandb run to track this script.
        run = wandb.init(
            # Set the wandb entity where your project will be logged (generally your team name).
            entity="21f2000660-dl-gen-ai-project-26-t1",
            # Set the wandb project where this run will be logged.
            project="21f2000660-t12026",
            # Track hyperparameters and run metadata.
            config={
                "learning_rate": 0.02,
                "architecture": "CNN",
                "dataset": "CIFAR-100",
                "epochs": 10,
            },
        )
    
    # Simulate training.
    epochs = 10
    offset = random.random() / 5
    for epoch in range(2, epochs):
        acc = 1 - 2**-epoch - random.random() / epoch - offset
        loss = 2**-epoch + random.random() / epoch + offset
    
        # Log metrics to wandb.
        run.log({"acc": acc, "loss": loss})
    
    # Finish the run and upload any remaining data.
    run.finish()

# Milestone 1

This milestone focuses on understanding the dataset and establishing a baseline performance through **exploratory data analysis (EDA)** and simple **heuristic-based methods** using `librosa`.

---

## Suggested Readings
- [Hugging Face Audio Course](https://huggingface.co/learn/audio-course/en/chapter0/introduction)
- [Librosa Documentation](https://librosa.org/doc/main/core.html#audio-loading)

---

## Instructions
Use this notebook to answer **all Milestone-1 questions**.

---

## Resources
- Notebook Link:  
  https://colab.research.google.com/drive/1m6UczhxQIke_raWSqukSWuiKbIVt7MMb?usp=sharing  

- Competition Link:  
  https://www.kaggle.com/competitions/jan-2026-dl-gen-ai-project/


In [8]:
if execute_milestone_solutions_code:
    #----------------------------- DON'T CHANGE THIS --------------------------
    DATA_SEED = 67
    TRAINING_SEED = 1234
    SR = 22050
    DURATION = 5.0
    N_FFT = 2048
    HOP_LENGTH = 512
    N_MELS = 128
    TOP_DB=20
    TARGET_SNR_DB = 10
    
    random.seed(DATA_SEED)
    np.random.seed(DATA_SEED)
    torch.manual_seed(DATA_SEED)
    torch.cuda.manual_seed(DATA_SEED)

In [9]:
if execute_milestone_solutions_code:
    # CONFIGURATION
    path = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems'
    DATA_ROOT = path # Enter dataset path
    
    with os.scandir(path) as entries:
        directories = [entry.name for entry in entries]
    
    GENRES = sorted(directories) # Make the list of all genres available (alphabetical order)
    print(GENRES)
    
    STEMS = {'drums':'drums.wav', 'vocals':'vocals.wav', 'bass':'bass.wav', 'other':'other.wav'} # Write here stems file name
    print(STEMS)
    
    STEM_KEYS = ['drums', 'vocals', 'bass', 'other']
    GENRE_TO_TEST = 'rock'
    SONG_INDEX = 0 #Enter index as per Q10.

In [10]:
if execute_milestone_solutions_code:
    def build_dataset(root_dir, val_split=0.17, seed=42):
        # Initialize empty dictionaries
        train_dataset = {g: {s.replace('.wav', ''): [] for s in STEMS} for g in GENRES}
        val_dataset   = {g: {s.replace('.wav', ''): [] for s in STEMS} for g in GENRES}
    
        #print(train_dataset)
    
        rng = random.Random(seed)
    
        # ------------------- write your code here -------------------------------
    
            # Iterate through Genres
            # Check: if genre folder exists
            # CHECK : Completeness (Does it have all stems?)
            # CHECK : Corruption (Is any file too small? (less than 4kb))
            # size checks
            # Stratified Shuffle Split
         #-------------------------------------------------------------------------
    
        # Limits given in the questions
        limit_1 = 4 * 1024
        limit_2 = 5.0491 * 1024 * 1024
        limit_3 = 5.0493 * 1024 * 1024
    
        corrupted_songs = 0
        small_songs = 0
        large_songs = 0
        
        # Helper function to populate dict
        def add_to_dict(target_dict, genre, song_paths):
            for song_path in song_paths:
                for stem_key, stem_file in STEMS.items():
                    target_dict[genre][stem_key].append(os.path.join(song_path, stem_file))
    
        for genre in GENRES:
            genre_path = os.path.join(root_dir, genre)
            
            # Check: if genre folder exists
            if not os.path.isdir(genre_path):
                continue
                
            # Get all song folders
            song_folders = sorted([f.path for f in os.scandir(genre_path) if f.is_dir()])
            valid_songs = []
    
            #print(song_folders[:3])
    
            for song in song_folders:
                is_valid = True
                
                # All stems for a song should be available, if not invalid 
                for stem_file in STEMS.values():
                    file_path = os.path.join(song, stem_file)
                    
                    if not os.path.exists(file_path):
                        is_valid = False
                        continue
                    
                    file_size = os.path.getsize(file_path)
                    
                    if file_size < limit_1:
                        corrupted_songs += 1
                        is_valid = False
                    
                    if file_size < limit_2:
                        small_songs += 1
                        
                    if file_size > limit_3:
                        large_songs += 1
    
                if is_valid:
                    valid_songs.append(song)
    
            # Stratified Shuffle Split
            rng.shuffle(valid_songs)
            
            split_index = int(len(valid_songs) * (1 - val_split))
            train_paths = valid_songs[:split_index]
            val_paths = valid_songs[split_index:]
            
            add_to_dict(train_dataset, genre, train_paths)
            add_to_dict(val_dataset, genre, val_paths)
    
        print('Q1',corrupted_songs + small_songs)
        print('Q2',abs(large_songs - small_songs))
        print('Q3',abs(len(train_dataset['reggae']['drums']) - len(val_dataset['country']['vocals'])))
    
        return train_dataset, val_dataset
    
    tr, val = build_dataset(DATA_ROOT)
    
    #print(val[GENRE_TO_TEST])

In [11]:
if execute_milestone_solutions_code:
    def find_long_silences(dataset_dict, sr=SR, threshold_sec=DURATION, top_db=TOP_DB):
        """
        Input:
            dataset_dict: The dictionary structure {genre: {stem: [paths...]}}
        Output:
            df: Pandas DataFrame containing details of all files with silence >= 5s
        """
        records = []
        # ------------------- write your code here -------------------------------
    
        total_files = 0    # ---- COUNT TOTAL FILES ----
    
    
    
            # Load Audio
    
            # Find Non-Silent Intervals
    
            # CASE A: Fully silent
            # CASE B: START silence
            # CASE C: END silence
            # CASE D: MIDDLE silence
    
            # Store result
            # if max_silence >= threshold_sec:
            #     records.append({
            #         "Genre": genre,
            #         "Stem": stem_name,
            #         "Duration": round(total_duration, 2),
            #         "Max_Silence_Sec": round(max_silence, 2),
            #         "Silence_Location": ", ".join(silence_type),
            #         "File_Path": file_path
            #     })
        #-------------------------------------------------------------------------
    
        # Iterate through the nested dictionary
        for genre, stems in dataset_dict.items():
            for stem_name, file_paths in stems.items():
                for file_path in file_paths:
                    total_files += 1
                    
                    # Load Audio
                    try:
                        y, sr = librosa.load(file_path, sr=sr)
                        total_duration = librosa.get_duration(y=y, sr=sr)
                    except Exception as e:
                        print(f"Error loading {file_path}: {e}")
                        continue
    
                    # Find Non-Silent Intervals
                    # intervals is a list of [start, end] sample indices of NON-silence
                    intervals = librosa.effects.split(y, top_db=top_db)
                    
                    silence_durations = []
                    silence_type = []
    
                    # CASE A: Fully silent
                    if len(intervals) == 0:
                        max_silence = total_duration
                        silence_type.append("Full")
                    else:
                        # CASE B: START silence (from 0 to first non-silent start)
                        if intervals[0][0] > 0:
                            start_silence = intervals[0][0] / sr
                            silence_durations.append(start_silence)
                            if start_silence >= threshold_sec:
                                 silence_type.append("Start")
    
                        # CASE C: END silence (from last non-silent end to total length)
                        if intervals[-1][1] < len(y):
                            end_silence = (len(y) - intervals[-1][1]) / sr
                            silence_durations.append(end_silence)
                            if end_silence >= threshold_sec:
                                 silence_type.append("End")
    
                        # CASE D: MIDDLE silence (gaps between intervals)
                        for i in range(len(intervals) - 1):
                            silence_gap = (intervals[i+1][0] - intervals[i][1]) / sr
                            silence_durations.append(silence_gap)
                            if silence_gap >= threshold_sec:
                                 silence_type.append("Middle")
                        
                        # Determine max silence for this file
                        if silence_durations:
                            max_silence = max(silence_durations)
                        else:
                            max_silence = 0.0
    
                    # Store result
                    if max_silence >= threshold_sec:
                        records.append({
                            "Genre": genre,
                            "Stem": stem_name,
                            "Duration": round(total_duration, 2),
                            "Max_Silence_Sec": round(max_silence, 4),
                            "Silence_Location": ", ".join(set(silence_type)), # use set to avoid duplicates
                            "File_Path": file_path
                        })
        df = pd.DataFrame(records)
        return df
    
    
    # --- EXECUTION ---
    # Pass your 'tr' (training) dictionary here.
    # Ensure 'tr' is defined from your previous build_dataset code.
    df_silence = find_long_silences(tr, threshold_sec=DURATION, top_db=TOP_DB)
    
    # --- RESULTS ANALYSIS ---
    
    # ------------------- write your code here -------------------------------
    
    print('Q4',len(df_silence))
    print('Q5',len(df_silence[df_silence['Stem'] == 'vocals']))
    print('Q6',df_silence[df_silence['Stem'] == 'vocals']['Max_Silence_Sec'].mean())
    print('Q7',len(df_silence[(df_silence['Genre'] == 'jazz') & (df_silence['Stem'] == 'drums')]))
    print('Q8',len(df_silence[(df_silence['Genre'] == 'jazz') & (df_silence['Stem'] == 'drums') & (df_silence['Silence_Location'] == 'Middle')]))
    print('Q9',len(df_silence[(df_silence['Genre'] == 'jazz') & (df_silence['Stem'] == 'drums') & (df_silence['Max_Silence_Sec'] >= 10.0)]))
    #-------------------------------------------------------------------------
    # Hint: Create a pivot Table: Count by Genre vs Stem
    
    pivot_table = df_silence.pivot_table(index='Genre', columns='Stem', values='Max_Silence_Sec', aggfunc='count', fill_value=0)
    print("\n--- Pivot Table ---")
    print(pivot_table)

In [12]:
if execute_milestone_solutions_code:
    stems_audio = []
    try:
        for key in STEM_KEYS:
        # ------------------- write your code here -------------------------------
        # Load audio (Duration 5.0s for speed/consistency)
            file_path = tr[GENRE_TO_TEST][key][SONG_INDEX]
    
            y, sr = librosa.load(file_path, sr=SR, duration=DURATION)
            stems_audio.append(y)
        #-------------------------------------------------------------------------
    
        print("Audio loaded successfully.")
    except NameError:
        print("ERROR: 'tr' dictionary not found. Please run build_dataset() first.")
    except IndexError:
        print(f"ERROR: Song index {SONG_INDEX} out of range for genre {GENRE_TO_TEST}.")
    except Exception as e:
        print(f"ERROR: {e}")

In [13]:
if execute_milestone_solutions_code:
    # ------------------- write your code here -------------------------------
    # Stack them into a numpy array (Shape: 4 x Samples)
    stems_stack = np.vstack(stems_audio)
    
    # Mix the stems by summing them element-wise
    mix_raw = np.sum(stems_stack, axis=0)
    
    # Calculate RMS Amplitude MANUALLY
    rms_val = np.sqrt(np.mean(mix_raw**2))
    
    #Peak Normalization
    max_val = np.max(np.abs(mix_raw))
    
    if max_val > 0:
        mix_norm = mix_raw / max_val
    else:
        mix_norm = mix_raw
    
    # VALIDATION
    assert np.isclose(np.max(np.abs(mix_norm)), 1.0), "Normalization failed."
    #------------------------------------------------------------------------
    
    print('Q10',len(mix_raw))
    print('Q11',rms_val)
    print('Q12',max_val)

# Milestone 2

In [14]:
if execute_milestone_solutions_code:
    #Mean duration of Jazz stems

    jazz_durations = []
    for stem_dict in tr.get('jazz').values():
        for file_path in stem_dict:
            dur = librosa.get_duration(path=file_path)
            jazz_durations.append(dur)
    
    mean_jazz_dur = np.mean(jazz_durations)
    print('Q1',mean_jazz_dur)

In [15]:
if execute_milestone_solutions_code:
    #Unique sample rates
    
    unique_srs = set()
    for genre, stems in tr.items():
        for stem_name, file_paths in stems.items():
            for file_path in file_paths:
                file_sr = librosa.get_samplerate(file_path)
                unique_srs.add(file_sr)
    
    print('Q2', unique_srs)

In [16]:
if execute_milestone_solutions_code:
    #Corrupted or zero-byte files
    
    zero_byte_count = 0
    for genre, stems in tr.items():
        for stem_name, file_paths in stems.items():
            for file_path in file_paths:
                if os.path.getsize(file_path) == 0:
                    zero_byte_count += 1
    
    print('Q3',zero_byte_count)

In [17]:
if execute_milestone_solutions_code:
    #Average peak amplitude for vocal stems
    
    vocal_peak_dbs = []
    for genre, stems in tr.items():
        vocal_files = stems.get('vocals')
        for file_path in vocal_files:
            y, sr = librosa.load(file_path, sr=SR)
            peak_amp = np.max(np.abs(y))
            peak_db = 20 * np.log10(max(peak_amp, 1e-10))
            vocal_peak_dbs.append(peak_db)
    
    avg_vocal_peak_db = np.mean(vocal_peak_dbs)
    print('Q4',avg_vocal_peak_db)

In [18]:
if execute_milestone_solutions_code:
    #Spectral Centroids
    
    genre_mean_centroids = {}
    
    for genre, stems in tr.items():
        genre_centroids = []
        for stem_name, file_paths in stems.items():
            for file_path in file_paths:
                y, _ = librosa.load(file_path, sr=SR)
                # Compute spectral centroid
                sc = librosa.feature.spectral_centroid(y=y, sr=SR)
                # Take the mean centroid of this specific file
                genre_centroids.append(np.mean(sc))
                
        if genre_centroids:
            genre_mean_centroids[genre] = np.mean(genre_centroids)
    
    print(genre_mean_centroids)
    
    blues_centroid = genre_mean_centroids.get('blues')
    print('Q5',blues_centroid)
    
    highest_sc_genre = max(genre_mean_centroids, key=genre_mean_centroids.get)
    print('Q6',highest_sc_genre)

In [19]:
if execute_milestone_solutions_code:
    #Silence in the first 0.5 seconds
    silence_count = 0
    for genre, stems in tr.items():
        for stem_name, file_paths in stems.items():
            for file_path in file_paths:
                y_start, sr = librosa.load(file_path, sr=SR, duration=0.5)
                
                intervals = librosa.effects.split(y_start, top_db=TOP_DB)
                
                if len(intervals) == 0 or intervals[0][0] > 0:
                    silence_count += 1
    
    print('Q7',silence_count)

# Programming Question

Complete the code as instructed in comments and answer the questions that follow.

In [20]:
if execute_milestone_solutions_code:
    # --- 1. Setup and Preprocessing ---
    ROOT = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup'
    STEMS_PATH = os.path.join(ROOT, 'genres_stems')
    GENRES = ["blues", "classical", "country", "disco", "hiphop", "jazz", "metal", "pop", "reggae", "rock"]
    
    def extract_features(song_path):
        # Load 10s at 22050Hz
        y, sr = librosa.load(os.path.join(song_path, 'other.wav'), sr=22050, duration=10)
        tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
        spec_cent = np.mean(librosa.feature.spectral_centroid(y=y, sr=sr))
        zcr = np.mean(librosa.feature.zero_crossing_rate(y))
        rolloff = np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr))
        return [float(tempo), spec_cent, zcr, rolloff]
    
    # --- 2. Data Preparation & Stratified Split ---
    data = []
    for g in GENRES:
        gp = os.path.join(STEMS_PATH, g)
        songs = [s for s in os.listdir(gp) if os.path.isdir(os.path.join(gp, s))]
        for s in songs[:50]: # Sampling 50 for speed; use all for final
            data.append({'path': os.path.join(gp, s), 'genre': g})
    
    df = pd.DataFrame(data)
    train_df, val_df = train_test_split(df, test_size=0.2, stratify=df['genre'], random_state=42)
    
    # --- 3. Model Training (Decision Tree) ---
    X_train = np.array([extract_features(p) for p in train_df['path']])
    y_train = train_df['genre']
    X_val = np.array([extract_features(p) for p in val_df['path']])
    y_val = val_df['genre']
    
    clf = DecisionTreeClassifier(max_depth=5, random_state=42)
    clf.fit(X_train, y_train)
    
    '''
    YOUR CODE HERE
    
    y_pred = # COMPUTE PREDICTED VALUES
    macro_f1 = # COMPUTE VALIDATION MACRO F1 SCORE
    cm = # COMPUTE CONFUSION MATRIX
    cr = # COMPUTE CLASSIFICATION REPORT
    
    '''
    
    y_pred = clf.predict(X_val)
    macro_f1 = f1_score(y_val, y_pred, average='macro')
    cm = confusion_matrix(y_val, y_pred, labels=GENRES)
    cr = classification_report(y_val, y_pred, target_names=GENRES)
    
    print(f"Validation Macro F1 Score: {macro_f1:.4f}\n")
    print("Detailed Classification Report:")
    print(cr)

In [21]:
if execute_milestone_solutions_code:
    
    '''
    YOUR CODE HERE
    
    Visualize the confusion matrix and compute TP, TN, FP, FN for all genres.
    '''
    # Visualize Confusion Matrix
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=GENRES, yticklabels=GENRES)
    plt.title('Confusion Matrix: Decision Tree')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()
    
    print('\n')
    print('--- TP, TN, FP, FN for all genres  ---')
    for i,genre in enumerate(GENRES):
        tp=cm[i, i]
        fp=cm[:, i].sum()-tp
        fn=cm[i, :].sum()-tp
        tn=cm.sum()-(tp+fp+fn)
            
        print(genre,"\t\t",tp,tn,fp,fn)

# Milestone 4

## Provided Helper Functions

In [22]:
if execute_milestone_solutions_code:
    # directory paths
    
    INPUT_BASE = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup'
    WORKING_BASE = '/kaggle/working'
    
    STEMS_PATH = os.path.join(INPUT_BASE, 'genres_stems')
    NOISE_PATH = os.path.join(INPUT_BASE, 'ESC-50-master/audio')
    OUTPUT_PATH = os.path.join(WORKING_BASE, 'synthetic_mashups/train')

In [23]:
if execute_milestone_solutions_code:
    # provided helper function 1
    
    def seed_everything(seed=42):
        """Locks all random seeds for absolute reproducibility."""
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        
        # If using GPU
        if torch.cuda.is_available():
            torch.cuda.manual_seed(seed)
            torch.cuda.manual_seed_all(seed)
            # Forces deterministic algorithms
            torch.backends.cudnn.deterministic = True 
            torch.backends.cudnn.benchmark = False
    
    # Execute immediately at the top of the script
    seed_everything(42)

In [24]:
if execute_milestone_solutions_code:
    # provided helper function 2
    
    def generate_synthetic_dataset(stems_dir, noise_dir, output_dir, samples_per_genre=50, target_sr=22050, duration=30):
        """Generates deterministic noisy mashups and saves them to /kaggle/working/."""
        genres = ["blues", "classical", "country", "disco", "hiphop", "jazz", "metal", "pop", "reggae", "rock"]
        target_length = target_sr * duration
        
        # Get noise files from read-only input
        noise_files = glob.glob(os.path.join(noise_dir, '**', '*.wav'), recursive=True)
        
        for genre in genres:
            # Create output directories in the writable /kaggle/working/ directory
            genre_out_dir = Path(output_dir) / genre
            genre_out_dir.mkdir(parents=True, exist_ok=True)
            
            song_folders = glob.glob(os.path.join(stems_dir, genre, '*'))
            if not song_folders: 
                print(f"Warning: No songs found for genre {genre}")
                continue
            
            for i in range(samples_per_genre):
                chosen_songs = random.sample(song_folders, 4)
                stems = []
                stem_types = ['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']
                
                for song, stem_type in zip(chosen_songs, stem_types):
                    stem_path = os.path.join(song, stem_type)
                    if os.path.exists(stem_path):
                        waveform, sr = torchaudio.load(stem_path)
                        
                        # Basic Resampling check (if needed)
                        if sr != target_sr:
                            resampler = torchaudio.transforms.Resample(sr, target_sr)
                            waveform = resampler(waveform)
    
                        if waveform.shape[1] > target_length:
                            waveform = waveform[:, :target_length]
                        elif waveform.shape[1] < target_length:
                            waveform = torch.nn.functional.pad(waveform, (0, target_length - waveform.shape[1]))
                        stems.append(waveform)
                
                if len(stems) == 4:
                    mashup = torch.stack(stems).sum(dim=0)
                    mashup = mashup / (torch.max(torch.abs(mashup)) + 1e-8)
                    
                    noise_file = random.choice(noise_files)
                    noise, _ = torchaudio.load(noise_file)
                    
                    if noise.shape[1] > target_length:
                        noise = noise[:, :target_length]
                        
                    start_idx = random.randint(0, target_length - noise.shape[1])
                    intensity = random.uniform(0.1, 0.4)
                    
                    mashup[:, start_idx:start_idx + noise.shape[1]] += (noise * intensity)
                    mashup = mashup / (torch.max(torch.abs(mashup)) + 1e-8)
                    
                    # Save to /kaggle/working/
                    out_path = genre_out_dir / f"mashup_{i:03d}.wav"
                    torchaudio.save(str(out_path), mashup, target_sr)
    
    # Run the generation
    generate_synthetic_dataset(STEMS_PATH, NOISE_PATH, OUTPUT_PATH, samples_per_genre=50)

In [25]:
if execute_milestone_solutions_code:
    # provided helper function 3
    
    def extract_and_save_features(input_dir, output_dir, target_sr=22050):
        """Converts audio to Mel-spectrograms in dB and saves as PyTorch tensors."""
        mel_transform = torchaudio.transforms.MelSpectrogram(
            sample_rate=target_sr, n_fft=2048, hop_length=512, n_mels=128
        )
        amplitude_to_db = torchaudio.transforms.AmplitudeToDB()
    
        # Find all .wav files in the input directory
        wav_files = glob.glob(os.path.join(input_dir, '**', '*.wav'), recursive=True)
        
        if not wav_files:
            print(f"Warning: No .wav files found in {input_dir}")
            return
    
        for wav_path in wav_files:
            # Replicate directory structure
            rel_path = os.path.relpath(wav_path, input_dir)
            out_path = Path(output_dir) / rel_path
            out_path = out_path.with_suffix('.pt')
            
            # Ensure the target directory exists in /kaggle/working/
            out_path.parent.mkdir(parents=True, exist_ok=True)
            
            # Process and save
            waveform, sr = torchaudio.load(wav_path)
            mel_spec = mel_transform(waveform)
            mel_spec_db = amplitude_to_db(mel_spec)
            
            torch.save(mel_spec_db, out_path)
        
        print(f"Successfully saved {len(wav_files)} feature files to {output_dir}")
    
    
    INPUT_DIR = '/kaggle/working/synthetic_mashups/train'
    OUTPUT_DIR = '/kaggle/working/features/train'
    
    extract_and_save_features(INPUT_DIR, OUTPUT_DIR)

In [26]:
if execute_milestone_solutions_code:
    waveform, sr = torchaudio.load('/kaggle/working/synthetic_mashups/train/blues/mashup_000.wav')
    print("Q2", tuple(waveform.shape))

In [27]:
if execute_milestone_solutions_code:
    pf_tensor = torch.load('/kaggle/working/features/train/blues/mashup_000.pt')
    print("Q3",tuple(pf_tensor.shape))

In [28]:
if execute_milestone_solutions_code:
    class PrecomputedFeatureDataset(Dataset):
        def __init__(self, features_dir):
            self.files = glob.glob(os.path.join(features_dir, '**', '*.pt'), recursive=True)
            self.genres = sorted(['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock'])
            self.genre_to_idx = {g: i for i, g in enumerate(self.genres)}
    
        def __len__(self):
            return len(self.files)
    
        def __getitem__(self, idx):
            file_path = self.files[idx]
            # Extract genre from the directory name
            genre = Path(file_path).parent.name
            label = self.genre_to_idx[genre] # converts genre name to a numerically encoded value
            
            # Load precomputed tensor
            feature = torch.load(file_path)
            return feature, label

In [29]:
if execute_milestone_solutions_code:
    class CRNN(nn.Module):
        def __init__(self, num_classes=10):
            super().__init__()
            
            # TODO 1: Define the CNN Backbone using nn.Sequential
            # Expect an input of shape (Batch_Size, 1, 128, Time_Steps)
            # Block 1: Conv2d(1 -> 32, kernel=3, padding=1) -> BatchNorm2d -> ReLU -> MaxPool2d(2)
            # Block 2: Conv2d(32 -> 64, kernel=3, padding=1) -> BatchNorm2d -> ReLU -> MaxPool2d(2)
            self.cnn = nn.Sequential(
                nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1),
                nn.BatchNorm2d(32),
                nn.ReLU(),
                nn.MaxPool2d(kernel_size=2),
                nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
                nn.BatchNorm2d(64),
                nn.ReLU(),
                nn.MaxPool2d(kernel_size=2)
            )
            
            # TODO 2: Define the RNN Component
            # Hint: Calculate the flattened feature size coming out of your CNN.
            # Original Mels = 128. After two MaxPool2d(2) layers, Mels = 128 / 4 = 32.
            # Channels = 64. So, Input Size = Channels * Mels = 2048.
            # Create a 1-layer Bidirectional LSTM with hidden_size=64 and batch_first=True.
            self.lstm = nn.LSTM(
                input_size=2048, 
                hidden_size=64, 
                num_layers=1, 
                batch_first=True, 
                bidirectional=True
            )
            
            # TODO 3: Define the Classifier
            # Create a Fully Connected (Linear) layer. 
            # Hint: Since the LSTM is bidirectional, the input features will be hidden_size * 2.
            self.fc = nn.Linear(in_features=128, out_features=num_classes)
    
        def forward(self, x):
            """
            Input:
                x: Tensor of shape (Batch, 1, 128, Time) representing Mel-spectrograms.
            Output:
                logits: Tensor of shape (Batch, num_classes) representing unnormalized class scores.
            """
            
            # TODO 4: Pass 'x' through the CNN backbone
            # Expected shape after CNN: (Batch, Channels=64, Mels=32, Time)
            x = self.cnn(x)
            
            # TODO 5: Bridge the gap (Shape Manipulation)
            # Permute and reshape the CNN output to be compatible with the LSTM.
            # Extract b, c, f, t from the tensor shape.
            # Permute the dimensions to bring Time forward, then reshape to flatten Channels and Mels.
            # Target shape for LSTM: (Batch, Time_Steps, Channels * Mels)
            b, c, f, t = x.shape
            print("Q4",tuple(x.shape))
            x = x.permute(0, 3, 1, 2)
            x = x.reshape(b, t, c * f)
            
            # TODO 6: Pass the reshaped sequence through the LSTM
            # The LSTM returns output and (hidden_state, cell_state). You only need the output.
            lstm_out, _ = self.lstm(x)
            
            # TODO 7: Global Max Pooling over the time dimension
            # Collapse the sequence down to a single vector using torch.max() over the time dimension (dim=1).
            # Note: torch.max returns both values and indices. You only need the values.
            pooled_out, _ = torch.max(lstm_out, dim=1)
            
            # TODO 8: Pass the pooled vector through the fully connected layer
            logits = self.fc(pooled_out)
            
            return logits

In [30]:
if execute_milestone_solutions_code:
    crnn_model = CRNN(num_classes=10)
    
    pf_tensor_dataset = PrecomputedFeatureDataset(OUTPUT_DIR)
    train_loader = DataLoader(pf_tensor_dataset, batch_size=32)
    real_features, real_labels = next(iter(train_loader))
    dummy_input = torch.randn(32, 1, 128, 1292)
    crnn_model(dummy_input)
    
    lstm_params = sum(p.numel() for p in temp_model.lstm.parameters() if p.requires_grad)
    print("Q5",lstm_params)

# Milestone 5

In [31]:
if execute_milestone_solutions_code:
    generate_synthetic_dataset(STEMS_PATH, NOISE_PATH, OUTPUT_PATH, samples_per_genre=100)

In [32]:
if execute_milestone_solutions_code:
    file_path = '/kaggle/working/synthetic_mashups/train/blues/mashup_000.wav'
    y, sr = librosa.load(file_path, sr=16000, duration=10)
    print(f"Q2: {y.shape}")

In [33]:
if execute_milestone_solutions_code:
    checkpoint = "MIT/ast-finetuned-audioset-10-10-0.4593"
    extractor = AutoFeatureExtractor.from_pretrained(checkpoint)
    
    # dummy audio as per the question
    dummy_mix = np.ones(160000)
    inputs = extractor(dummy_mix, sampling_rate=16000, return_tensors="pt")
    
    # Squeeze(0)
    ast_input = inputs['input_values'].squeeze(0)
    print(f"Q3: {ast_input.shape}")

In [34]:
if execute_milestone_solutions_code:
    model = ASTForAudioClassification.from_pretrained(
        checkpoint,
        num_labels=10,
        ignore_mismatched_sizes=True
    )
    
    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Q4 : {total_params}")

In [35]:
if execute_milestone_solutions_code:
    y_test = np.array([-0.85, 0.40, 0.20, -0.10])
    y = y_test / (np.max(np.abs(y_test)) + 1e-9)
    print(f"Q5 : {y}")

# DummyModel Submission for Registration

In [36]:
if execute_milestone_solutions_code:
    
    testData = pd.read_csv("/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/test.csv")
    path = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems"
    
    with os.scandir(path) as entries:
        directories = [entry.name for entry in entries]
    
    ids = [f"{i:04d}" for i in range(1, len(testData)+1)]
    genres = random.choices(directories, k=3020)
    
    submission = pd.DataFrame({
        'id': ids,
        'genre' : genres
    })
    
    submission.head()
    
    submission.to_csv('/kaggle/working/submission.csv', index=False)

# Submission using the DecisionTreeClassifier model

In [37]:
if execute_milestone_solutions_code:
    TEST_CSV_PATH = os.path.join(ROOT, 'test.csv')
    
    def extract_features(song_path):
        # Load 10s at 22050Hz
        y, sr = librosa.load(song_path, sr=22050, duration=10)
        tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
        spec_cent = np.mean(librosa.feature.spectral_centroid(y=y, sr=sr))
        zcr = np.mean(librosa.feature.zero_crossing_rate(y))
        rolloff = np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr))
        return [float(tempo), spec_cent, zcr, rolloff]
    
    test_df = pd.read_csv(TEST_CSV_PATH)
    
    #print(test_df.shape)
    #print(test_df.sample(5))
    
    X_test = []
    
    for index, row in test_df.iterrows():
        file_name = row['filename']
        file_path = os.path.join(ROOT, file_name)
        features = extract_features(file_path)
        X_test.append(features)
    
    X_test = np.array(X_test)
    test_pred = clf.predict(X_test)
    
    submission = pd.DataFrame({
        'id': test_df['id'],
        'genre' : test_pred
    })
    
    submission.head()
    
    submission.to_csv('/kaggle/working/submission.csv', index=False)

# Final Submission

In [38]:
def seed_everything(seed=42):
    """Locks all random seeds for absolute reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    
    # If using GPU
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        # Forces deterministic algorithms
        torch.backends.cudnn.deterministic = True 
        torch.backends.cudnn.benchmark = False

# Execute immediately at the top of the script
seed_everything(42)

In [39]:
INPUT_BASE = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup'
WORKING_BASE = '/kaggle/working'

STEMS_PATH = os.path.join(INPUT_BASE, 'genres_stems')
NOISE_PATH = os.path.join(INPUT_BASE, 'ESC-50-master/audio')
TEST_CSV = os.path.join(INPUT_BASE, 'test.csv')
TEST_AUDIO_DIR = os.path.join(INPUT_BASE, 'mashups')
SYNTHETIC_TRAIN_DIR = os.path.join(WORKING_BASE, 'synthetic_mashups/train')
SYNTHETIC_VAL_DIR = os.path.join(WORKING_BASE, 'synthetic_mashups/val')

In [40]:
song_path = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/mashups/song0001.wav'
y, sr = librosa.load(song_path)
print(f"The sampling rate of test audio data is: {sr} Hz")
total_duration = librosa.get_duration(y=y, sr=sr)
print(f"The duration of test audio data is: {total_duration} sec")

The sampling rate of test audio data is: 22050 Hz
The duration of test audio data is: 30.00018140589569 sec


In [41]:
# # 1. Skip the generation step, just grab the paths from the working directory!
# PREVIOUS_VERSION_PATH = '/kaggle/input/notebooks/gokulvasudevans/dl-21f2000660-notebook-t12026/synthetic_mashups/train/**/*.wav'
# saved_files = glob.glob(PREVIOUS_VERSION_PATH, recursive=True)

# # 2. Rebuild the file_list dictionary expected by the Dataset class
# reusable_file_list = []
# for path in saved_files:
#     # Extract genre from the folder name (e.g., .../train/blues/set1_000.wav -> 'blues')
#     genre = path.split('/')[-2] 
#     reusable_file_list.append({'path': path, 'genre': genre})

# print(f"Loaded {len(reusable_file_list)} files directly from previously saved files!")

In [42]:
GENRES = sorted(["blues", "classical", "country", "disco", "hiphop", 
                 "jazz", "metal", "pop", "reggae", "rock"])
GENRE_TO_IDX = {g: i for i, g in enumerate(GENRES)}
IDX_TO_GENRE = {i: g for i, g in enumerate(GENRES)}

SR = 16000
DURATION = 10
TARGET_SAMPLES = SR * DURATION
SAMPLES_PER_SET = 100 # 100 * 5 sets * 10 genres = 5000

## Data Validation and Visualization

In [43]:
def validate_data(stems_path, noise_path):
    print("Validating dataset to remove corrupted or silent files...")
    valid_stems = {g: [] for g in GENRES}
    
    for genre in GENRES:
        folders = glob.glob(os.path.join(stems_path, genre, '*'))
        for folder in folders:
            is_valid = True
            for stem in ['drums.wav', 'bass.wav', 'vocals.wav', 'other.wav']:
                p = os.path.join(folder, stem)
                # Check if file exists and isn't practically empty
                if not os.path.exists(p) or os.path.getsize(p) < 4*1024:
                    is_valid = False
                    break
                    
            if is_valid: 
                valid_stems[genre].append(folder)
            
    raw_noise = glob.glob(os.path.join(noise_path, '**', '*.wav'), recursive=True)
    valid_noise = [n for n in raw_noise if os.path.getsize(n) > 1000]
    
    print(f"Validation complete! Found {len(valid_noise)} valid noise files.")
    return valid_stems, valid_noise

def visualize_spectrogram(audio_tensor, title="Mel Spectrogram"):
    mel_transform = T.MelSpectrogram(sample_rate=SR, n_fft=1024, hop_length=512, n_mels=128)
    to_db = T.AmplitudeToDB()
    mel_spec_db = to_db(mel_transform(audio_tensor)).squeeze().numpy()
    
    plt.figure(figsize=(10, 4))
    librosa.display.specshow(mel_spec_db, sr=SR, hop_length=512, x_axis='time', y_axis='mel', cmap='magma')
    plt.colorbar(format="%+2.f dB")
    plt.title(title)
    plt.tight_layout()
    plt.show()

## Messy Mashup Data Generation

In [44]:
def tempo_sync_and_mix(stems_dict, noise_path=None, sync_tempo=True):
    """Loads, tempo-syncs to the drum track, pads/truncates, and mixes."""
    loaded_stems = {}
    for stem_name, path in stems_dict.items():
        y, _ = librosa.load(path, sr=SR)
        loaded_stems[stem_name] = y
        
    mix = np.zeros(TARGET_SAMPLES)
    master_tempo = 0
    
    if sync_tempo:
        drum_track = loaded_stems.get('drums.wav', loaded_stems['vocals.wav'])
        master_tempo, _ = librosa.beat.beat_track(y=drum_track, sr=SR)
        master_tempo = float(master_tempo)
    
    for stem_name, y in loaded_stems.items():
        # Apply time stretching only if it's not the master track and tempo > 0
        if sync_tempo and stem_name != 'drums.wav' and master_tempo > 0:
            stem_tempo, _ = librosa.beat.beat_track(y=y, sr=SR)
            stem_tempo = float(stem_tempo)
            if stem_tempo > 0:
                rate = np.clip(master_tempo / stem_tempo, 0.7, 1.3) # Prevent extreme chipmunking
                y = librosa.effects.time_stretch(y, rate=rate)
        
        # Pad or Truncate
        if len(y) > TARGET_SAMPLES:
            y = y[:TARGET_SAMPLES] 
        else:
            y = np.pad(y, (0, TARGET_SAMPLES - len(y)))
        mix += y
        
    # Inject ESC-50 Noise
    if noise_path:
        n_wave, _ = librosa.load(noise_path, sr=SR)
        if len(n_wave) > TARGET_SAMPLES:
            n_wave = n_wave[:TARGET_SAMPLES] 
        else:
            n_wave = np.pad(n_wave, (0, TARGET_SAMPLES - len(n_wave)))
        mix += (n_wave * random.uniform(0.1, 0.4))
        
    # Max-Normalization to prevent clipping
    mix = mix / (np.max(np.abs(mix)) + 1e-9) 
    return torch.tensor(mix, dtype=torch.float32).unsqueeze(0)

def split_stems_prevent_leakage(valid_stems):
    print("Splitting raw song folders to prevent data leakage (Perfectly Stratified)...")
    train_stems = {g: [] for g in GENRES}
    val_stems = {g: [] for g in GENRES}
    
    all_folders = []
    all_labels = []
    
    # 1. Flatten the dictionary into a single list of folders and their corresponding labels
    for genre in GENRES:
        for folder in valid_stems[genre]:
            all_folders.append(folder)
            all_labels.append(genre)
            
    # 2. Perform a single global split using the 'stratify' parameter
    # This guarantees the 80/20 ratio is perfectly maintained for every genre class
    tr_f, val_f, tr_y, val_y = train_test_split(
        all_folders, 
        all_labels, 
        test_size=0.2, 
        random_state=42, 
        stratify=all_labels  # <--- The magic parameter
    )
    
    # 3. Re-pack the split data back into the dictionary format the generator expects
    for folder, genre in zip(tr_f, tr_y):
        train_stems[genre].append(folder)
        
    for folder, genre in zip(val_f, val_y):
        val_stems[genre].append(folder)
        
    return train_stems, val_stems

def build_dataset(valid_stems, valid_noise, target_dir, samples_per_set):
    print("Generating Tempo-Synced 5-Set Curriculum Dataset...")
    os.makedirs(target_dir, exist_ok=True)
    generated_files = []
    
    for genre in tqdm(GENRES, desc="Processing Genres"):
        genre_out = os.path.join(target_dir, genre)
        os.makedirs(genre_out, exist_ok=True)
        songs = valid_stems[genre]
        
        if len(songs) < 4: continue
        
        file_idx = 0
            
        for i in range(samples_per_set):
            base_folder = random.choice(songs)
            base_stems = {s: os.path.join(base_folder, s) for s in ['drums.wav', 'bass.wav', 'vocals.wav', 'other.wav']}
            
            # Helper to save and record with Set ID
            def save_and_record(mix_tensor, genre_label, out_path, idx, set_id):
                filename = f"{idx:03d}.wav"
                full_path = os.path.join(out_path, filename)
                torchaudio.save(full_path, mix_tensor, SR)
                return {'path': full_path, 'genre': genre_label, 'set_id': set_id}

            # --- SET 1: Clean Baseline ---
            mix1 = tempo_sync_and_mix(base_stems, noise_path=None, sync_tempo=False)
            generated_files.append(save_and_record(mix1, genre, genre_out, file_idx, set_id=1))
            file_idx += 1
            
            # --- SET 2: Noisy Baseline ---
            mix2 = tempo_sync_and_mix(base_stems, noise_path=random.choice(valid_noise), sync_tempo=False)
            generated_files.append(save_and_record(mix2, genre, genre_out, file_idx, set_id=2))
            file_idx += 1
            
            # --- SET 3: Intra-Genre Mashup Clean ---
            s3_f = random.sample(songs, 4)
            s3_stems = { 'drums.wav': os.path.join(s3_f[0], 'drums.wav'), 'bass.wav': os.path.join(s3_f[1], 'bass.wav'),
                         'vocals.wav': os.path.join(s3_f[2], 'vocals.wav'), 'other.wav': os.path.join(s3_f[3], 'other.wav')}
            mix3 = tempo_sync_and_mix(s3_stems, noise_path=None, sync_tempo=True)
            generated_files.append(save_and_record(mix3, genre, genre_out, file_idx, set_id=3))
            file_idx += 1

            # --- SET 4: Intra-Genre Mashup Noisy ---
            mix4 = tempo_sync_and_mix(s3_stems, noise_path=random.choice(valid_noise), sync_tempo=True)
            generated_files.append(save_and_record(mix4, genre, genre_out, file_idx, set_id=4))
            file_idx += 1

            # --- SET 5: Cross-Genre Mashup Noisy ---
            s5_dom = random.sample(songs, 2)
            cross_g = random.sample([g for g in GENRES if g != genre], 2)
            s5_stems = {
                'vocals.wav': os.path.join(s5_dom[0], 'vocals.wav'),
                'bass.wav': os.path.join(s5_dom[1], 'bass.wav'),
                'drums.wav': os.path.join(random.choice(valid_stems[cross_g[0]]), 'drums.wav'),
                'other.wav': os.path.join(random.choice(valid_stems[cross_g[1]]), 'other.wav')
            }
            mix5 = tempo_sync_and_mix(s5_stems, noise_path=random.choice(valid_noise), sync_tempo=True)
            generated_files.append(save_and_record(mix5, genre, genre_out, file_idx, set_id=5))
            file_idx += 1
            
    return generated_files


def load_or_generate_data():
    SAVED_FILES_TRAIN_PATH = '/kaggle/input/notebooks/gokulvasudevans/dl-21f2000660-notebook-t12026/synthetic_mashups/train/**/*.wav'
    saved_train_files = glob.glob(SAVED_FILES_TRAIN_PATH, recursive=True)
    train_files = []
    
    SAVED_FILES_VAL_PATH = '/kaggle/input/notebooks/gokulvasudevans/dl-21f2000660-notebook-t12026/synthetic_mashups/val/**/*.wav'
    saved_val_files = glob.glob(SAVED_FILES_VAL_PATH, recursive=True)
    val_files = []

    if len(saved_train_files) > 0 and len(saved_val_files) > 0:
        print(f"Found {len(saved_train_files)} train files from previous version.")
        for path in saved_train_files:
            genre = path.split('/')[-2]
            filename = os.path.basename(path)
            idx = int(filename.replace('.wav', '').replace('mashup_', ''))
            set_id = (idx % 5) + 1 # From creation, we know that 0=Set1, 1=Set2, 2=Set3, 3=Set4, 4=Set5, 5=Set1...
                
            train_files.append({'path': path, 'genre': genre, 'set_id': set_id})

        print(f"Found {len(saved_val_files)} validation files from previous version.")
        for path in saved_val_files:
            genre = path.split('/')[-2]
            filename = os.path.basename(path)
            idx = int(filename.replace('.wav', '').replace('mashup_', ''))
            set_id = (idx % 5) + 1 # From creation, we know that 0=Set1, 1=Set2, 2=Set3, 3=Set4, 4=Set5, 5=Set1...
                
            val_files.append({'path': path, 'genre': genre, 'set_id': set_id})
    else:
        valid_stems, valid_noise = validate_data(STEMS_PATH, NOISE_PATH)
        train_stems, val_stems = split_stems_prevent_leakage(valid_stems)
        train_files = build_dataset(train_stems, valid_noise, SYNTHETIC_TRAIN_DIR, samples_per_set=80)
        val_files = build_dataset(val_stems, valid_noise, SYNTHETIC_VAL_DIR, samples_per_set=20)
        print(f"Training Samples: {len(train_files)}")
        print(f"Validation Samples: {len(val_files)}")
    
    return train_files, val_files

In [45]:
train_files, val_files = load_or_generate_data()

Validating dataset to remove corrupted or silent files...
Validation complete! Found 2000 valid noise files.
Splitting raw song folders to prevent data leakage (Perfectly Stratified)...
Generating Tempo-Synced 5-Set Curriculum Dataset...


Processing Genres: 100%|██████████| 10/10 [56:08<00:00, 336.82s/it]


Generating Tempo-Synced 5-Set Curriculum Dataset...


Processing Genres: 100%|██████████| 10/10 [13:00<00:00, 78.04s/it]

Training Samples: 4000
Validation Samples: 1000


## Custom Messy Mashup Dataset

In [46]:
# class SpecAugment(nn.Module):
#     def __init__(self, freq_mask=20, time_mask=40):
#         super().__init__()
#         self.aug = nn.Sequential(
#             T.FrequencyMasking(freq_mask_param=freq_mask),
#             T.TimeMasking(time_mask_param=time_mask)
#         )

#     def forward(self, x):
#         # Torchaudio masking expects [..., Freq, Time]
#         return self.aug(x)

In [47]:
# class MessyMashupDataset(Dataset):
#     def __init__(self, file_list, mode='ast', is_Train=False):
#         self.file_list = file_list
#         self.mode = mode
#         self.is_train = is_Train
#         self.mel_tf = T.MelSpectrogram(sample_rate=SR, n_fft=1024, hop_length=512, n_mels=128)
#         self.to_db = T.AmplitudeToDB()
#         self.augmenter = SpecAugment() if is_Train else None
        
#         if mode == 'ast':
#             self.ast_extractor = AutoFeatureExtractor.from_pretrained("MIT/ast-finetuned-audioset-10-10-0.4593")

#     def __len__(self): return len(self.file_list)

#     def __getitem__(self, idx):
#         path = self.file_list[idx]['path']
#         label = GENRE_TO_IDX[self.file_list[idx]['genre']]

#         waveform, _ = torchaudio.load(path)

#         # Apply Random Gain (Volume shift between 0.5x and 1.5x) ONLY during training
#         if self.is_train:
#             gain = random.uniform(0.5, 1.5)
#             waveform = waveform * gain
#             # Ensure it doesn't clip past -1.0 or 1.0
#             waveform = torch.clamp(waveform, -1.0, 1.0)
        
#         if self.mode in ['cnn', 'crnn']:
#             # Mel Transform creates shape: [1, Freq, Time]
#             spec = self.to_db(self.mel_tf(waveform))

#             if self.is_train:
#                 spec = self.augmenter(spec)
#         else:
#             # AST expects 1D NumPy arrays in shape: [Time, Freq]
#             inputs = self.ast_extractor(waveform.squeeze(0).numpy(), sampling_rate=SR, return_tensors="pt")
#             spec = inputs['input_values'].squeeze(0)
            
#             if self.is_train:
#                 # 1. Flip to [Freq, Time]
#                 # 2. Add fake channel [1, Freq, Time] for Torchaudio
#                 spec = spec.transpose(0, 1).unsqueeze(0) 
#                 spec = self.augmenter(spec)
#                 # 3. Strip fake channel and flip back to [Time, Freq] for AST
#                 spec = spec.squeeze(0).transpose(0, 1)
                
#         return spec, label

## Model Architectures

In [48]:
# class EfficientNetAudio(nn.Module):
#     def __init__(self, num_classes=10):
#         super().__init__()
#         self.efnet = models.efficientnet_b0(weights=None)
#         # Modify for 1-Channel audio
#         orig_conv = self.efnet.features[0][0]
#         self.efnet.features[0][0] = nn.Conv2d(1, 
#                                               orig_conv.out_channels, 
#                                               orig_conv.kernel_size, 
#                                               orig_conv.stride, 
#                                               orig_conv.padding, 
#                                               bias=False)
#         self.efnet.classifier[1] = nn.Linear(self.efnet.classifier[1].in_features, num_classes)
        
#     def forward(self, x): 
#         return self.efnet(x)

# class CRNN(nn.Module):
#     def __init__(self, num_classes=10):
#         super().__init__()
#         self.cnn = nn.Sequential(
#             nn.Conv2d(1, 32, 3, padding=1),
#             nn.BatchNorm2d(32),
#             nn.ReLU(),
#             nn.MaxPool2d(2),
#             nn.Conv2d(32, 64, 3, padding=1),
#             nn.BatchNorm2d(64),
#             nn.ReLU(),
#             nn.MaxPool2d(2)
#         )
#         self.lstm = nn.LSTM(2048, 64, batch_first=True, bidirectional=True)
#         self.fc = nn.Linear(128, num_classes)
#     def forward(self, x):
#         x = self.cnn(x)
#         b, c, f, t = x.shape
#         lstm_out, _ = self.lstm(x.permute(0, 3, 1, 2).reshape(b, t, c * f))
#         pooled, _ = torch.max(lstm_out, dim=1)
#         return self.fc(pooled)

# def get_ast_model():
#     return ASTForAudioClassification.from_pretrained(
#         "MIT/ast-finetuned-audioset-10-10-0.4593", 
#         num_labels=10, ignore_mismatched_sizes=True
#     )

## Training

In [49]:
# class EarlyStopping:
#     def __init__(self, patience=3, min_delta=0):
#         self.patience = patience
#         self.min_delta = min_delta
#         self.counter = 0
#         self.best_loss = float('inf')
#         self.early_stop = False

#     def __call__(self, val_loss):
#         if val_loss < self.best_loss - self.min_delta:
#             self.best_loss = val_loss
#             self.counter = 0
#         else:
#             self.counter += 1
#             print(f"EarlyStopping counter: {self.counter} out of {self.patience}")
#             if self.counter >= self.patience:
#                 self.early_stop = True

In [50]:
# def train_pipeline(model_name='ast', epochs=1, batch_size=8):    
#     # 1. Datasets and sample visualization
#     train_files, val_files = load_or_generate_data()
    
#     sample_wave, _ = torchaudio.load(train_files[0]['path'])
#     visualize_spectrogram(sample_wave, title="Train Data Sample Spectrogram : " + train_files[0]['genre'])
    
#     sample_wave, _ = torchaudio.load(val_files[0]['path'])
#     visualize_spectrogram(sample_wave, title="Val Data Sample Spectrogram : " + val_files[0]['genre'])

#     # 2. DataLoader
#     mode = 'ast' if model_name == 'ast' else model_name
    
#     train_loader = DataLoader(MessyMashupDataset(train_files, mode, True), batch_size=batch_size, shuffle=True)
#     val_loader = DataLoader(MessyMashupDataset(val_files, mode, False), batch_size=batch_size, shuffle=False)

#     # Convert the loader into an iterator and grab the first batch
#     # data_batch, label_batch = next(iter(val_loader))
    
#     # print(f"Batch Data Shape: {data_batch.shape}")   # e.g., [8, 1, 1024, 128] for AST
#     # print(f"Batch Labels: {label_batch}")           # e.g., tensor([3, 0, 5, ...])
    
#     # Select the first index
#     # single_file = data_batch[0]
#     # single_label = label_batch[0]
    
#     # print(f"Single File Shape: {single_file.shape}")
#     # print(f"Single File Genre Index: {single_label.item()}")
#     # print(f"Actual Genre Name: {IDX_TO_GENRE[single_label.item()]}")
    
#     # 3. Initialize Model & Hyperparameters
#     if model_name == 'cnn': model = EfficientNetAudio().to(device); lr = 1e-3
#     elif model_name == 'crnn': model = CRNN().to(device); lr = 1e-3
#     elif model_name == 'ast': model = get_ast_model().to(device); lr = 5e-5
    
#     criterion = nn.CrossEntropyLoss()
#     optimizer = optim.AdamW(model.parameters(), lr=lr)
    
#     # 4. Initialize W&B
#     wandb.init(project="21f2000660-dl-gen-ai-project-26-t1", name=f"{model_name}_training_run")
#     early_stopper = EarlyStopping(patience=3)
#     best_f1 = 0.0

#     for epoch in range(epochs):
#         # Training Phase
#         model.train()
#         train_loss = 0.0
#         for inputs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
#             inputs, labels = inputs.to(device), labels.to(device)
#             optimizer.zero_grad()
            
#             if model_name == 'ast':
#                 outputs = model(inputs, labels=labels)
#                 loss = outputs.loss
#             else:
#                 loss = criterion(model(inputs), labels)
                
#             loss.backward()
#             optimizer.step()
#             train_loss += loss.item()
            
#         # Evaluation Phase
#         model.eval()
#         all_preds, all_labels = [], []
#         with torch.no_grad():
#             for inputs, labels in val_loader:
#                 inputs, labels = inputs.to(device), labels.to(device)
#                 logits = model(inputs).logits if model_name == 'ast' else model(inputs)
#                 all_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
#                 all_labels.extend(labels.cpu().numpy())
                
#         # Metrics
#         mac_f1 = f1_score(all_labels, all_preds, average='macro')
#         acc = accuracy_score(all_labels, all_preds)
        
#         wandb.log({'Epoch': epoch+1, "Train Loss": train_loss/len(train_loader), "Val F1": mac_f1, "Val Acc": acc})
#         print(f"Epoch {epoch+1} | Loss: {train_loss/len(train_loader):.4f} | Val F1: {mac_f1:.4f}")
        
#         if mac_f1 > best_f1:
#             best_f1 = mac_f1
#             torch.save(model.state_dict(), f"/kaggle/working/best_{model_name}.pth")
#             print(f"New best model saved to /kaggle/working/best_{model_name}.pth")

#         early_stopper(mac_f1)
#         if early_stopper.early_stop:
#             print(f"Early stopping triggered. No improvement in Val F1 for {early_stopper.patience} epochs.")
#             break
            
#     wandb.finish()
#     return f"/kaggle/working/best_{model_name}.pth"

In [51]:
# def train_two_stage_pipeline(model_name='ast', phase1_epochs=2, phase2_epochs=5, batch_size=8):    
#     train_files, val_files = load_or_generate_data()

#     mode = 'ast' if model_name == 'ast' else model_name
    
#     train_loader = DataLoader(MessyMashupDataset(train_files, mode, is_Train=True), batch_size=batch_size, shuffle=True)
#     val_loader = DataLoader(MessyMashupDataset(val_files, mode, is_Train=False), batch_size=batch_size, shuffle=False)

#     sample_wave, _ = torchaudio.load(train_files[0]['path'])
#     visualize_spectrogram(sample_wave, title="Train Data Sample Spectrogram : " + train_files[0]['genre'])

#     sample_wave, _ = torchaudio.load(val_files[0]['path'])
#     visualize_spectrogram(sample_wave, title="Val Data Sample Spectrogram : " + val_files[0]['genre'])
    
#     model = get_ast_model().to(device)
#     criterion = nn.CrossEntropyLoss()
    
#     wandb.init(project="21f2000660-dl-gen-ai-project-26-t1", name=f"{model_name}_TwoStage_Run")
#     best_f1 = 0.0

#     # ==============================================================================
#     # PHASE 1: CLASSIFIER WARMUP (Freeze the Base)
#     # ==============================================================================
#     print("\n" + "="*50)
#     print("PHASE 1: Classifier Warmup (Freezing AST Base)")
#     print("="*50)
    
#     # Freeze the pre-trained transformer base, leave only the classifier unfrozen
#     for param in model.audio_spectrogram_transformer.parameters():
#         param.requires_grad = False
        
#     # Larger learning rate just for the classifier
#     optimizer_phase1 = optim.AdamW(model.classifier.parameters(), lr=1e-3)
    
#     for epoch in range(phase1_epochs):
#         model.train()
#         train_loss = 0.0
#         for inputs, labels in tqdm(train_loader, desc=f"Phase 1 - Epoch {epoch+1}"):
#             inputs, labels = inputs.to(device), labels.to(device)
#             optimizer_phase1.zero_grad()
#             outputs = model(inputs, labels=labels)
#             loss = outputs.loss
#             loss.backward()
#             optimizer_phase1.step()
#             train_loss += loss.item()
            
#         # Quick validation (No saving yet, just monitoring)
#         model.eval()
#         all_preds, all_labels = [], []
#         with torch.no_grad():
#             for inputs, labels in val_loader:
#                 logits = model(inputs.to(device)).logits
#                 all_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
#                 all_labels.extend(labels.numpy())
#         mac_f1 = f1_score(all_labels, all_preds, average='macro')
#         print(f"Phase 1 - Epoch {epoch+1} | Loss: {train_loss/len(train_loader):.4f} | Val F1: {mac_f1:.4f}")

#     # ==============================================================================
#     # PHASE 2: FULL FINE-TUNE (Unfreeze + Cosine Scheduler)
#     # ==============================================================================
#     print("\n" + "="*50)
#     print("PHASE 2: Full Fine-Tuning (Unfreezing All Layers)")
#     print("="*50)
    
#     # Unfreeze the base
#     for param in model.audio_spectrogram_transformer.parameters():
#         param.requires_grad = True
        
#     # Smaller learning rate for the whole model, with weight decay to prevent overfitting
#     optimizer_phase2 = optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
    
#     # Cosine Scheduler: Warms up for 10% of steps, then gently decays
#     total_steps = len(train_loader) * phase2_epochs
#     warmup_steps = int(0.1 * total_steps)
#     scheduler = get_cosine_schedule_with_warmup(optimizer_phase2, num_warmup_steps=warmup_steps, num_training_steps=total_steps)
    
#     early_stopper = EarlyStopping(patience=3)

#     for epoch in range(phase2_epochs):
#         model.train()
#         train_loss = 0.0
#         for inputs, labels in tqdm(train_loader, desc=f"Phase 2 - Epoch {epoch+1}"):
#             inputs, labels = inputs.to(device), labels.to(device)
#             optimizer_phase2.zero_grad()
#             outputs = model(inputs, labels=labels)
#             loss = outputs.loss
#             loss.backward()
#             optimizer_phase2.step()
#             scheduler.step() # Step the scheduler every batch!
#             train_loss += loss.item()
            
#         model.eval()
#         all_preds, all_labels = [], []
#         with torch.no_grad():
#             for inputs, labels in val_loader:
#                 logits = model(inputs.to(device)).logits
#                 all_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
#                 all_labels.extend(labels.numpy())
                
#         mac_f1 = f1_score(all_labels, all_preds, average='macro')
#         acc = accuracy_score(all_labels, all_preds)
        
#         current_lr = optimizer_phase2.param_groups[0]['lr']
#         wandb.log({'Epoch': epoch+1+phase1_epochs, "Train Loss": train_loss/len(train_loader), "Val F1": mac_f1, "LR": current_lr})
#         print(f"Phase 2 - Epoch {epoch+1} | Loss: {train_loss/len(train_loader):.4f} | Val F1: {mac_f1:.4f} | LR: {current_lr:.6f}")
        
#         if mac_f1 > best_f1:
#             best_f1 = mac_f1
#             torch.save(model.state_dict(), f"/kaggle/working/best_ast_twostage.pth")
#             print(f"New best Phase 2 model saved!")

#         early_stopper(mac_f1)
#         if early_stopper.early_stop:
#             print("Early stopping triggered in Phase 2.")
#             break
            
#     wandb.finish()
#     return f"/kaggle/working/best_ast_twostage.pth"

## Prediction

In [52]:
# CHOSEN_MODEL = 'ast'
# n_epochs = 5
# batch_size = 8

# test_df = pd.read_csv(TEST_CSV)

# if CHOSEN_MODEL == 'cnn': model = EfficientNetAudio().to(device)
# elif CHOSEN_MODEL == 'crnn': model = CRNN().to(device)
# elif CHOSEN_MODEL == 'ast': model = get_ast_model().to(device)

# if CHOSEN_MODEL == 'ast':
#     print(f"Prediction using {CHOSEN_MODEL} model with two stage training pipeline and batch size of {batch_size}")
#     best_weights = train_two_stage_pipeline(model_name='ast', phase1_epochs=2, phase2_epochs=5, batch_size=batch_size)
# else:
#     print(f"Prediction using {CHOSEN_MODEL} model with {n_epochs} epochs and batch size of {batch_size}")
#     best_weights = train_pipeline(model_name=CHOSEN_MODEL, epochs=n_epochs, batch_size=batch_size)
    
# model.load_state_dict(torch.load(best_weights, map_location=device))

# #best_ast_model_from_previous_version = '/kaggle/input/datasets/gokulvasudevans/best-ast-model-from-previous-training/best_ast.pth'
# #model.load_state_dict(torch.load(best_ast_model_from_previous_version))

# model.eval()

# mel_tf = T.MelSpectrogram(sample_rate=SR, n_fft=1024, hop_length=512, n_mels=128)
# to_db = T.AmplitudeToDB()
# ast_ext = AutoFeatureExtractor.from_pretrained("MIT/ast-finetuned-audioset-10-10-0.4593")

# CHUNK_DURATION = 10
# CHUNK_SAMPLES = SR * CHUNK_DURATION
# STEP_SAMPLES = SR * 5
# preds = []

# with torch.no_grad():
#     for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Predicting"):
#         path = os.path.join(TEST_AUDIO_DIR, f"song{row['id']:04d}.wav")
#         wave, sr = torchaudio.load(path)
        
#         # Test Preprocessing (Ensure exact matching to training)
#         if sr != SR: wave = T.Resample(sr, SR)(wave)
#         if wave.shape[0] > 1: wave = torch.mean(wave, dim=0, keepdim=True)
#         # wave = wave[:, :TARGET_SAMPLES] if wave.shape[1] > TARGET_SAMPLES else torch.nn.functional.pad(wave, (0, TARGET_SAMPLES - wave.shape[1]))
#         wave = wave / (torch.max(torch.abs(wave)) + 1e-9)
        
#         total_samples = wave.shape[1]
#         all_chunk_logits = []

#         # Step through the 30-second audio with a 5-second overlapping window
#         for start in range(0, total_samples, STEP_SAMPLES):
#             end = start + CHUNK_SAMPLES
#             chunk = wave[:, start:end]
            
#             # Zero-pad if we hit the end of the file and the chunk is less than 10s
#             if chunk.shape[1] < CHUNK_SAMPLES:
#                 chunk = torch.nn.functional.pad(chunk, (0, CHUNK_SAMPLES - chunk.shape[1]))
            
#             # Route to the correct Preprocessing & Model Forward Pass
#             if CHOSEN_MODEL == 'ast':
#                 # AST expects a 1D numpy array
#                 chunk_np = chunk.squeeze(0).numpy()
#                 inputs = ast_ext(chunk_np, sampling_rate=SR, return_tensors="pt")
#                 inputs = inputs['input_values'].to(device)
#                 logits = model(inputs).logits
            
#             else: # CNN or CRNN
#                 # They expect a 2D Mel-Spectrogram in DB [Batch, Channel, Freq, Time]
#                 chunk = chunk.to(device)
#                 spec = to_db(mel_tf(chunk)).unsqueeze(0) 
#                 logits = model(spec)
                
#             all_chunk_logits.append(logits)

#         # ENSEMBLE: Average the confidence scores from all overlapping chunks
#         final_logits = torch.mean(torch.stack(all_chunk_logits), dim=0)
#         final_pred = torch.argmax(final_logits, dim=1).item()
        
#         preds.append(IDX_TO_GENRE[final_pred])

In [53]:
# submission_df = pd.DataFrame({'id': test_df['id'], 'genre': preds})
# submission_df.to_csv('submission.csv', index=False)